# Procesador de Variables 
Modelo Multimoney (Entrada 0)

**Descripción General**

Este script procesa archivos JSON provenientes de consultas
de Buró de Crédito y extrae exclusivamente las variables
definidas como "Entrada 0" para el modelo Multimoney.

**Objetivo**

Generar un JSON limpio, anonimizado y listo para scoring,
eliminando completamente información sensible del solicitante.

**Variables Generadas**

• CREDMAXAUTCONS

• MONTOPAGARCONS

• SALDOACTCONS

• CREDMAXAUTREV

• SALDOACTREV

• LIMITECREDITO

• SALDOACTTDC

• CREDMAXAUTTOTAL

• SALDOVENCTOTAL

• FECHA

• FECHAULTCONSULTA

• PEORMOP

• PEOR_HIST_1 a PEOR_HIST_24

**Metodología de Cálculo**

1. Los montos se limpian eliminando símbolos "+" y ceros vacíos.
2. Los agregados se calculan por tipoContrato según Anexo 3 del Manual API Reporte de Crédito.
3. PEORMOP se obtiene como el máximo entre:
       - formaPagoActual
       - mopHistoricoMorosidadMasGrave
4. PEOR_HIST_1–24:
       - Se toman los últimos 24 caracteres de historicoPagos.
       - Se alinean desde el mes más reciente.
       - Se calcula el peor valor por posición entre todas las cuentas.


**Uso Esperado**

Este script forma parte del pipeline previo al modelo
predictivo de probabilidad de mora (90+ días).

**Salida**

Devuelve un JSON estructurado con:

    - id_solicitud anonimizado

    - Variables agregadas de Entrada 0

Listo para ser consumido por el modelo de scoring.


In [1]:
# Librerías
import json
import hashlib
import re
import numpy as np
from datetime import datetime

In [2]:
# Configuración Anexo 3
CONSUMO = ["AN", "AG", "AL", "AP", "AU", "BD", "BT", "CE", "CF", "CO", "CS", "CT", "DE", "EQ"
           , "FI", "FT", "HA", "HE", "HI", "LS", "MI", "OA", "PA", "PB", "PG", "PL","PN", "PQ"
           "PR", "PS", "RC", "RD", "RE", "RF", "RN", "RV", "SE", "SG", "SM", "ST", "UK", "US"]
REVOLVENTE = ["CL", "LR"]
TDC = ["CC", "SC", "TE"]


In [3]:
def limpiar_numero(valor):

    if valor is None or valor == "":
        return np.nan

    valor = str(valor).replace("+", "").strip()

    try:
        return int(valor)
    except:
        return np.nan


def generar_id_anonimo(json_data):

    try:
        base = json_data["respuesta"]["persona"]["encabezado"].get(
            "numeroReferenciaOperador", ""
        )
        return hashlib.sha256(base.encode()).hexdigest()[:16]

    except:
        return "id_no_disponible"


def cargar_json_seguro(path_json):

    with open(path_json, "r", encoding="utf-8") as f:
        contenido = f.read()

    try:
        return json.loads(contenido)

    except json.JSONDecodeError:

        contenido = re.sub(
            r'("valorScore"\s*:\s*"\d+")\s*("codigoRazon")',
            r'\1,\2',
            contenido
        )

        return json.loads(contenido)


def peor_mop(cuentas):

    peor = np.nan

    for c in cuentas:

        for campo in ["formaPagoActual", "mopHistoricoMorosidadMasGrave"]:

            mop = c.get(campo)

            if mop and str(mop).isdigit():

                mop = int(mop)

                if np.isnan(peor) or mop > peor:
                    peor = mop

    return peor


def fecha_ult_consulta(consultas):

    fechas = []

    for c in consultas:

        fecha = c.get("fechaConsulta")

        if fecha:
            try:
                fechas.append(datetime.strptime(fecha, "%d%m%Y"))
            except:
                pass

    if fechas:
        return max(fechas).strftime("%Y-%m-%d")

    return np.nan


def calcular_peor_historico_1_24(cuentas):

    peor_por_mes = [np.nan] * 24

    for c in cuentas:

        hist = str(c.get("historicoPagos", ""))

        if hist == "":
            continue

        ultimos_24 = hist[-24:].rjust(24, "0")

        for i in range(24):

            ch = ultimos_24[-(i + 1)]

            if ch.isdigit():

                ch = int(ch)

                if np.isnan(peor_por_mes[i]) or ch > peor_por_mes[i]:
                    peor_por_mes[i] = ch

    resultado = {}

    for i in range(24):
        resultado[f"PEOR_HIST_{i+1}"] = peor_por_mes[i]

    return resultado


def procesar_json(path_json):

    warnings = []

    data = cargar_json_seguro(path_json)

    try:
        persona = data["respuesta"]["persona"]
    except:
        raise ValueError("El JSON no contiene la estructura esperada del Buró")

    cuentas = persona.get("cuentas", [])
    consultas = persona.get("consultaEfectuadas", [])
    resumen = persona.get("resumenReporte", [{}])[0]

    if not cuentas:
        warnings.append("El JSON no contiene cuentas reportadas")

    if not consultas:
        warnings.append("El JSON no contiene historial de consultas")

    id_solicitud = generar_id_anonimo(data)

    CREDMAXAUTCONS = np.nan
    MONTOPAGARCONS = np.nan
    SALDOACTCONS = np.nan
    CREDMAXAUTREV = np.nan
    SALDOACTREV = np.nan
    LIMITECREDITO = np.nan
    SALDOACTTDC = np.nan
    CREDMAXAUTTOTAL = np.nan
    SALDOVENCTOTAL = np.nan

    for c in cuentas:

        tipo = c.get("tipoContrato")

        creditoMaximo = limpiar_numero(c.get("creditoMaximo"))
        saldoActual = limpiar_numero(c.get("saldoActual"))
        saldoVencido = limpiar_numero(c.get("saldoVencido"))
        limiteCredito = limpiar_numero(c.get("limiteCredito"))
        montoPagar = limpiar_numero(c.get("montoPagar"))

        if not np.isnan(creditoMaximo):
            CREDMAXAUTTOTAL = np.nansum([CREDMAXAUTTOTAL, creditoMaximo])

        if not np.isnan(saldoVencido):
            SALDOVENCTOTAL = np.nansum([SALDOVENCTOTAL, saldoVencido])

        if tipo in CONSUMO:

            CREDMAXAUTCONS = np.nansum([CREDMAXAUTCONS, creditoMaximo])
            MONTOPAGARCONS = np.nansum([MONTOPAGARCONS, montoPagar])
            SALDOACTCONS = np.nansum([SALDOACTCONS, saldoActual])

        if tipo in REVOLVENTE:

            CREDMAXAUTREV = np.nansum([CREDMAXAUTREV, creditoMaximo])
            SALDOACTREV = np.nansum([SALDOACTREV, saldoActual])
            LIMITECREDITO = np.nansum([LIMITECREDITO, limiteCredito])

        if tipo in TDC:

            SALDOACTTDC = np.nansum([SALDOACTTDC, saldoActual])

    resultado = {
        "id_solicitud": id_solicitud,
        "CREDMAXAUTCONS": CREDMAXAUTCONS,
        "MONTOPAGARCONS": MONTOPAGARCONS,
        "SALDOACTCONS": SALDOACTCONS,
        "CREDMAXAUTREV": CREDMAXAUTREV,
        "SALDOACTREV": SALDOACTREV,
        "LIMITECREDITO": LIMITECREDITO,
        "SALDOACTTDC": SALDOACTTDC,
        "CREDMAXAUTTOTAL": CREDMAXAUTTOTAL,
        "SALDOVENCTOTAL": SALDOVENCTOTAL,
        "FECHA": resumen.get("fechaSolicitudReporteMasReciente", np.nan),
        "FECHAULTCONSULTA": fecha_ult_consulta(consultas),
        "PEORMOP": peor_mop(cuentas)
    }

    resultado.update(calcular_peor_historico_1_24(cuentas))

    return resultado, warnings

In [4]:
# Ejecución
if __name__ == "__main__":

    input_path = "json/RCO_Int_007_Hawk_Response.json"
    output_path = "output_modelo_multimoney.json"

    resultado = procesar_json(input_path)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(resultado, f, indent=4)

    print(json.dumps(resultado, indent=4))

[
    {
        "id_solicitud": "8c74ac0ef524bfe9",
        "CREDMAXAUTCONS": 1060020.0,
        "MONTOPAGARCONS": 23730.0,
        "SALDOACTCONS": 596709.0,
        "CREDMAXAUTREV": 24758.0,
        "SALDOACTREV": 548.0,
        "LIMITECREDITO": 41656.0,
        "SALDOACTTDC": 55587.0,
        "CREDMAXAUTTOTAL": 1324645.0,
        "SALDOVENCTOTAL": 112469.0,
        "FECHA": "27012026",
        "FECHAULTCONSULTA": "2026-01-27",
        "PEORMOP": 96,
        "PEOR_HIST_1": 6,
        "PEOR_HIST_2": 7,
        "PEOR_HIST_3": 7,
        "PEOR_HIST_4": 7,
        "PEOR_HIST_5": 7,
        "PEOR_HIST_6": 7,
        "PEOR_HIST_7": 7,
        "PEOR_HIST_8": 7,
        "PEOR_HIST_9": 7,
        "PEOR_HIST_10": 7,
        "PEOR_HIST_11": 9,
        "PEOR_HIST_12": 9,
        "PEOR_HIST_13": 9,
        "PEOR_HIST_14": 7,
        "PEOR_HIST_15": 7,
        "PEOR_HIST_16": 7,
        "PEOR_HIST_17": 7,
        "PEOR_HIST_18": 7,
        "PEOR_HIST_19": 7,
        "PEOR_HIST_20": 7,
        "PEOR